LTSM  

In [6]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from sklearn.metrics import classification_report
train_df = pd.read_csv('sent_train.csv')
valid_df = pd.read_csv('sent_valid.csv')
# Cleaning the text data by removing URLs, special characters, and converting to lowercase
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text
train_df['clean_text'] = train_df['text'].apply(clean_text)
valid_df['clean_text'] = valid_df['text'].apply(clean_text)

In [7]:
# Hyperparameters 
# Maximum number of words to keep based on word frequency in the dataset
MAX_NB_WORDS = 10000 
# Maximum length of sequences after padding
MAX_SEQUENCE_LENGTH = 100
# Dimension of the embedding vector for each word
EMBEDDING_DIM = 64
# Tokenization and padding of the text data
tokenizer = Tokenizer(num_words=MAX_NB_WORDS)
tokenizer.fit_on_texts(train_df['clean_text'].values)
# Convert text to sequences and pad them to ensure uniform input length for the model
X_train = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_text'].values), maxlen=MAX_SEQUENCE_LENGTH)
X_valid = pad_sequences(tokenizer.texts_to_sequences(valid_df['clean_text'].values), maxlen=MAX_SEQUENCE_LENGTH)
# Extract labels
Y_train = train_df['label'].values
Y_valid = valid_df['label'].values

In [ ]:
# Builduing the LSTM Model for Text Classification
model = Sequential([
    # Transforms word indices into dense vectors with a specified dimension
    Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    # Regularization technique for NLP to prevent overfitting by dropping entire 1D feature maps
    SpatialDropout1D(0.2),
    # LSTM layer to handle sequential data  with dropout for regularization
    LSTM(64, dropout=0.2),
    # Output layer with Softmax for 3-class classification cause we have 3 output classes
    Dense(3, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# Train the model
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_data=(X_valid, Y_valid), verbose=1)
plots = model.history.history
# Evaluate the model on the validation set
Y_pred = model.predict(X_valid)
Y_pred_classes = np.argmax(Y_pred, axis=1)
print(classification_report(Y_valid, Y_pred_classes))

Epoch 1/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.6667 - loss: 0.8164 - val_accuracy: 0.7328 - val_loss: 0.6373
Epoch 2/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.7790 - loss: 0.5354 - val_accuracy: 0.8011 - val_loss: 0.5230
Epoch 3/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.8710 - loss: 0.3458 - val_accuracy: 0.8229 - val_loss: 0.4830
Epoch 4/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.9208 - loss: 0.2343 - val_accuracy: 0.8291 - val_loss: 0.5136
Epoch 5/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 7s 46ms/step - accuracy: 0.9471 - loss: 0.1628 - val_accuracy: 0.8254 - val_loss: 0.6013
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
              precision    recall  f1-score   support

           0       0.73      0.54      0.62       347
           1       0.75      0.69      0.72       475
           2       0.86      0.93      0.89      1566

    accuracy                           0.83      2388
   macro avg       0.78      0.72      0.74     